Signals from the features analysis are used to train a model to predict the target variable. The model can be a machine learning algorithm such as a decision tree, random forest, or neural network. The trained model can then be used to make predictions on new data.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
df = pd.read_csv(r'C:\Users\Opeyemi\OneDrive - UNC Kenan-Flagler Business School\Desktop\Data Science\Fraudulent-Transaction-Detection-for-Digital-Money-Transfer/data/cleaned_transactionsV1.csv')

In [13]:
df.columns.tolist()

['timestamp',
 'home_country',
 'source_currency',
 'dest_currency',
 'channel',
 'amount_src',
 'amount_usd',
 'fee',
 'exchange_rate_src_to_dest',
 'new_device',
 'ip_country',
 'location_mismatch',
 'ip_risk_score',
 'kyc_tier',
 'account_age_days',
 'device_trust_score',
 'chargeback_history_count',
 'risk_score_internal',
 'txn_velocity_1h',
 'txn_velocity_24h',
 'corridor_risk',
 'is_fraud',
 'hour_of_day',
 'day_of_week',
 'is_weekend',
 'is_night',
 'corridor',
 'age_bucket',
 'amount_bucket',
 'ip_risk_bucket',
 'device_trust_bucket',
 'night_hours',
 'account_very_new',
 'account_new',
 'velocity_burst',
 'amount_high',
 'ip_risk_high',
 'device_low']

In [14]:
print(df.columns.tolist())

['timestamp', 'home_country', 'source_currency', 'dest_currency', 'channel', 'amount_src', 'amount_usd', 'fee', 'exchange_rate_src_to_dest', 'new_device', 'ip_country', 'location_mismatch', 'ip_risk_score', 'kyc_tier', 'account_age_days', 'device_trust_score', 'chargeback_history_count', 'risk_score_internal', 'txn_velocity_1h', 'txn_velocity_24h', 'corridor_risk', 'is_fraud', 'hour_of_day', 'day_of_week', 'is_weekend', 'is_night', 'corridor', 'age_bucket', 'amount_bucket', 'ip_risk_bucket', 'device_trust_bucket', 'night_hours', 'account_very_new', 'account_new', 'velocity_burst', 'amount_high', 'ip_risk_high', 'device_low']


In [16]:
#Drop temporary buckets and select final features
df = df.drop(['age_bucket','amount_bucket', 'ip_risk_bucket', 'device_trust_bucket']\
             , axis=1)

In [18]:
#define features set
categorical_features = ['channel','kyc_tier', 'home_country', 'source_currency', 'dest_currency',
                         'ip_country', 'new_device', 'location_mismatch']

numerical_features = ['amount_src', 'amount_usd', 'fee', 'ip_risk_score', 'device_trust_score', 
                      'account_age_days', 'txn_velocity_1h', 'txn_velocity_24h', 'corridor_risk', 
                      'risk_score_internal', 'hour', 'day_of_week', 'is_weekend', 'night_hours', 
                      'account_very_new', 'account_new', 'velocity_burst', 'amount_high', 'ip_high_risk', 'device_low_trust']

all_features = categorical_features + numerical_features
print(f'total features: {len(all_features)}')
print(f'categorical features: {len(categorical_features)}')
print(f'numerical features: {len(numerical_features)}')
print(f'\nDataframe shape: {df.shape}')


total features: 28
categorical features: 8
numerical features: 20

Dataframe shape: (11200, 34)


## Threshold-Based Features

- `night_hours` : 1 if transaction between 3–7 AM, else 0
- `account_very_new` : 1 if account <30 days old
- `account_new` : 1 if account 30–90 days old
- `velocity_burst` : 1 if ≥3 transactions in last hour
- `amount_high` : 1 if amount ≥2000 USD
- `ip_high_risk` : 1 if IP risk score >0.8
- `device_low_trust` : 1 if device trust score <0.5

These convert continuous or categorical signals into simple binary flags highlighting potential fraud risk.

In [19]:
# Based on our analysis, create threshold-based features
df['night_hours'] = ((df['hour_of_day'] >= 3) & (df['hour_of_day'] <= 7)).astype(int)
df['account_very_new'] = (df['account_age_days'] < 30).astype(int)
df['account_new'] = ((df['account_age_days'] >= 30) & (df['account_age_days'] < 90)).astype(int)
df['velocity_burst'] = (df['txn_velocity_1h'] >= 3).astype(int)
df['amount_high'] = (df['amount_usd'] >= 2000).astype(int)
df['ip_high_risk'] = (df['ip_risk_score'] > 0.8).astype(int)
df['device_low_trust'] = (df['device_trust_score'] < 0.5).astype(int)

print("Features created:")
print(df[['night_hours', 'account_very_new', 'velocity_burst', 'amount_high',
          'ip_high_risk', 'device_low_trust']].describe())

Features created:
        night_hours  account_very_new  velocity_burst   amount_high  \
count  11200.000000      11200.000000    11200.000000  11200.000000   
mean       0.216875          0.170179        0.095268      0.027589   
std        0.412135          0.375806        0.293598      0.163800   
min        0.000000          0.000000        0.000000      0.000000   
25%        0.000000          0.000000        0.000000      0.000000   
50%        0.000000          0.000000        0.000000      0.000000   
75%        0.000000          0.000000        0.000000      0.000000   
max        1.000000          1.000000        1.000000      1.000000   

       ip_high_risk  device_low_trust  
count  11200.000000      11200.000000  
mean       0.129375          0.242054  
std        0.335630          0.428346  
min        0.000000          0.000000  
25%        0.000000          0.000000  
50%        0.000000          0.000000  
75%        0.000000          0.000000  
max        1.000000   

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11200 entries, 0 to 11199
Data columns (total 36 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   timestamp                  11200 non-null  str    
 1   home_country               11200 non-null  str    
 2   source_currency            11200 non-null  str    
 3   dest_currency              11200 non-null  str    
 4   channel                    11200 non-null  str    
 5   amount_src                 11200 non-null  float64
 6   amount_usd                 11200 non-null  float64
 7   fee                        11200 non-null  float64
 8   exchange_rate_src_to_dest  11200 non-null  float64
 9   new_device                 11200 non-null  bool   
 10  ip_country                 11200 non-null  str    
 11  location_mismatch          11200 non-null  bool   
 12  ip_risk_score              11200 non-null  float64
 13  kyc_tier                   11200 non-null  str    
 14  a

: 